# Residual-stream learning phases in deep hierarchical models

This notebook replaces the head-centric question in `head_phases.ipynb` with a depth-aware one:

> **At which training step does each teacher latent become linearly decodable from each residual stream?**

The axes are the ones emitted by `ProbeLogger`: residual `L0…LN`, teacher `level0…levelD−1`, within-unit `slot`, and relative-unit offset `k`. `L0` is the post input/position encoder baseline; `L1…LN` are the residuals after transformer blocks 1…N. No attention-head assignment is assumed.

The analysis logic lives in `src/analysis/residual_phases.py`; this notebook is its interactive shell. Run it from the repository root.

In [ ]:
%load_ext autoreload
%autoreload 2

import pathlib, sys, warnings

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.analysis.residual_phases import (
    PROBE_METRICS, acquisition_table, adjacent_layer_deltas,
    layer_order_by_target, missing_probe_cells, probe_history_long,
    probe_metric_key, probe_metric_keys, probe_spec_from_config,
    target_order_by_layer,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 140)
sns.set_theme(style="whitegrid")

## 1. Configure

`DEMO=True` makes the notebook fully executable without W&B. For a real run, set it to `False` and enter the run ID.

The headline analysis defaults to `k=-1` (retention of the previous unit) and `k=0` (belief about the current unit). Those regimes have logged teacher Bayes ceilings. `k=+1` planning can still be plotted, but its ceiling is not currently logged and should not be mixed into the main cross-level ordering.

In [ ]:
DEMO = True

ENTITY, PROJECT = "okyksl", "hmm-attention"  # conf/misc/default.yaml
RUN_ID = "replace-me"
SPLIT = "val"

# Acquisition = start of the final run of MIN_DWELL evaluations at or above
# THRESHOLD of the uniform-to-Bayes NLL improvement range.
THRESHOLD = 0.80
MIN_DWELL = 3
REQUIRE_FINAL = True
ANALYZE_OFFSETS = [-1, 0]

# One raw-metric detail view: (teacher level, slot, offset).
DETAIL_TARGET = (0, 0, 0)

## 2. Load one run and resolve its probe grid

The synthetic example deliberately makes different residuals win for different teacher levels; it is a format check, not a scientific prediction. Real dimensions—number of blocks, hierarchy arities, class counts, and offsets—are inferred from the resolved run config.

In [ ]:
def make_demo_run():
    config = {
        "teacher": {
            "levels": [
                {"chunk_dim": 5, "chunk_size": 2},
                {"chunk_dim": 9, "chunk_size": 3},
            ],
            "base_teacher": {"dim": 7, "window": 1, "span_lengths": [1]},
        },
        "student": {"num_blocks": 3},
        "misc": {"probe": {"mode": "warm_start", "offsets": [-1, 0, 1]}},
        "dataset": {"window": 1},
    }
    spec = probe_spec_from_config(config)
    steps = np.arange(0, 2100, 100)
    columns = {"_step": steps}

    # Layer-specific onsets by (teacher level, regime). Later slots receive
    # slightly earlier onsets because more of the current unit is observable.
    onset = {
        (0, -1): [1050, 700, 350, 550],
        (0,  0): [1250, 850, 300, 500],
        (0,  1): [1700, 1450, 1000, 1150],
        (1, -1): [850, 250, 450, 650],
        (1,  0): [1000, 200, 400, 600],
        (1,  1): [1550, 900, 1100, 1300],
    }
    for level, num_slots in enumerate(spec.slots_per_level):
        classes = spec.class_counts[level]
        chance = 1 / classes
        for slot in range(num_slots):
            for offset in spec.offsets:
                for layer in range(spec.num_layers):
                    center = onset[(level, offset)][layer] - 60 * slot
                    progress = 1 / (1 + np.exp(-(steps - center) / 120))
                    progress = np.clip(progress + 0.012 * np.sin(steps / 170 + layer), 0, 1)
                    if offset < 0:
                        ceiling, bayes_nll = 1.0, 0.0
                    elif offset == 0:
                        slot_fraction = slot / max(1, num_slots - 1)
                        ceiling = chance + (1 - chance) * (0.48 + 0.20 * slot_fraction)
                        bayes_nll = max(0.05, -np.log(ceiling))
                    else:
                        ceiling, bayes_nll = 0.88, np.nan

                    acc = chance + progress * (ceiling - chance)
                    acc_key = probe_metric_key(layer, level, slot, offset, "acc", SPLIT)
                    columns[acc_key] = acc
                    columns[probe_metric_key(layer, level, slot, offset, "n", SPLIT)] = 1000
                    nll = (bayes_nll + (1 - progress) * (np.log(classes) - bayes_nll)
                           if np.isfinite(bayes_nll) else np.log(classes) - progress)
                    columns[probe_metric_key(layer, level, slot, offset, "nll", SPLIT)] = nll
                    if offset <= 0:
                        columns[probe_metric_key(layer, level, slot, offset, "bayes_acc", SPLIT)] = ceiling
                        columns[probe_metric_key(layer, level, slot, offset, "bayes_nll", SPLIT)] = bayes_nll
                        columns[probe_metric_key(layer, level, slot, offset, "excess_nll", SPLIT)] = nll - bayes_nll
    history = pd.DataFrame(columns)
    return config, spec, history, "synthetic-depth-demo"


if DEMO:
    config, spec, history, run_name = make_demo_run()
else:
    from notebooks.utils import fetch_run_config, fetch_run_data

    config = fetch_run_config(RUN_ID, entity=ENTITY, project=PROJECT)
    spec = probe_spec_from_config(config)
    payload = fetch_run_data(
        RUN_ID, probe_metric_keys(spec, PROBE_METRICS, SPLIT),
        entity=ENTITY, project=PROJECT,
    )
    history, run_name = payload["df"], payload["name"]

probe_mode = config.get("misc", {}).get("probe", {}).get("mode", "off")
if probe_mode == "off":
    raise ValueError("This run has probe logging disabled.")
if probe_mode == "sgd":
    warnings.warn(
        "SGD probes co-train with the student: acquisition time mixes representation "
        "formation with probe-optimizer lag. Prefer warm_start runs for learning-order claims."
    )

layout = pd.DataFrame({
    "teacher_level": range(spec.num_levels),
    "slots": spec.slots_per_level,
    "latent_classes": spec.class_counts,
    "surface_span": spec.level_spans,
})
print(f"{run_name}: residuals L0…L{spec.num_layers - 1}; offsets {spec.offsets}; probe_mode={probe_mode}")
display(layout)

## 3. Make probe quality comparable across depths

Raw accuracy is misleading across teacher levels: chance is $1/C_\ell$, and the current-unit target can be intrinsically uncertain. The primary score uses the logger's Bayes-aligned excess NLL:

$$q = 1 - \frac{\mathrm{excessNLL}}{\log C_\ell - \mathrm{BayesNLL}}.$$

`q=0` matches a uniform predictor and `q=1` reaches the teacher's Bayes NLL. `excess_nll` compares probe and Bayes on the same valid rows, which makes it cleaner than accuracy for current-unit refinement. Overshoots remain visible. Chance-to-Bayes accuracy is retained as a diagnostic column. For planning (`k>0`) no Bayes ceiling is logged, so the notebook excludes it from headline ordering by default.

In [ ]:
missing = missing_probe_cells(history, spec, SPLIT)
if len(missing):
    print(f"Warning: {len(missing)} expected accuracy cells are absent; they will be skipped.")
    display(missing.head(10))

probe_long = probe_history_long(history, spec, split=SPLIT, strict=False)
coverage = (
    probe_long.groupby(["level", "slot", "offset", "ceiling_source"])
    .agg(cells=("layer", "nunique"), evaluations=("_step", "nunique"))
    .reset_index()
)
display(coverage)

## 4. Trajectories before declaring phases

One panel is one semantic target `(teacher level, slot, offset)`; lines are residual streams. Do not average slots here: for `k=0`, later slots contain more observations and are expected to have a different Bayes ceiling.

In [ ]:
def plot_progress(offset):
    max_slots = max(spec.slots_per_level)
    fig, axes = plt.subplots(
        spec.num_levels, max_slots,
        figsize=(4.1 * max_slots, 3.0 * spec.num_levels),
        sharex=True, sharey=True, squeeze=False,
    )
    palette = sns.color_palette("colorblind", spec.num_layers)
    for level in range(spec.num_levels):
        for slot in range(max_slots):
            ax = axes[level, slot]
            if slot >= spec.slots_per_level[level]:
                ax.set_visible(False)
                continue
            cell = probe_long[(probe_long.level == level) &
                              (probe_long.slot == slot) &
                              (probe_long.offset == offset)]
            for layer, trajectory in cell.groupby("layer"):
                ax.plot(trajectory["_step"], trajectory["nll_progress"],
                        color=palette[layer], linewidth=2, label=f"L{layer}")
            ax.axhline(THRESHOLD, color="0.35", linestyle="--", linewidth=1)
            ax.set_title(f"level{level} / slot{slot} / k={offset:+d}")
            ax.set_ylim(-0.1, 1.15)
            ax.spines[["top", "right"]].set_visible(False)
            if level == spec.num_levels - 1:
                ax.set_xlabel("training step")
        axes[level, 0].set_ylabel("uniform→Bayes NLL progress")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=spec.num_layers, frameon=False)
    fig.suptitle(f"{run_name}: probe trajectories for k={offset:+d}", y=1.02)
    fig.tight_layout()
    plt.show()


for offset in ANALYZE_OFFSETS:
    if offset in spec.offsets:
        plot_progress(offset)

## 5. Convert curves into stable acquisition events

Acquisition is the start of the **final** run of at least `MIN_DWELL` logged evaluations above `THRESHOLD`. This suppresses early lucky spikes. Times are only resolved to the probe logging cadence; equal times are reported as ties.

The first table answers “which residual gets this datum first?” The second answers “which data reaches this residual first?”

In [ ]:
headline_offsets = [offset for offset in ANALYZE_OFFSETS if offset in spec.offsets]
if not headline_offsets:
    raise ValueError(f"None of ANALYZE_OFFSETS={ANALYZE_OFFSETS} were logged: {spec.offsets}")

events = acquisition_table(
    probe_long, score="nll_progress", threshold=THRESHOLD,
    min_dwell=MIN_DWELL, require_final=REQUIRE_FINAL,
    offsets=headline_offsets,
)
by_target = layer_order_by_target(events)
by_residual = target_order_by_layer(events)

print("Residual order for every latent target")
display(by_target[["target", "regime", "first_layer", "first_step", "layer_order"]])
print("Latent-target order inside every residual")
display(by_residual[["residual", "first_target", "first_step", "target_order"]])

## 6. First-arrival map

Rows are latent targets; columns are residual streams; color and annotation are acquisition steps. Blank cells were never stably acquired under the chosen criterion. This is the depth-aware analogue of the phase table in `head_phases.ipynb`.

In [ ]:
target_order = by_target["target"].tolist()
arrival = (
    events.pivot(index="target", columns="layer", values="acquisition_step")
    .reindex(target_order)
    .rename(columns=lambda layer: f"L{layer}")
)
fig, ax = plt.subplots(figsize=(1.2 * spec.num_layers + 3, 0.48 * len(arrival) + 2))
sns.heatmap(arrival, annot=True, fmt=".0f", cmap="mako_r", linewidths=0.5,
            cbar_kws={"label": "stable acquisition step"}, ax=ax)
ax.set_title(f"{run_name}: when each residual makes each latent decodable")
ax.set_xlabel("residual stream")
ax.set_ylabel("teacher latent target")
fig.tight_layout()
plt.show()

## 7. Adjacent residual lead/lag

For adjacent residuals, $\Delta_{j,i}=t_{L_j}-t_{L_i}$. Negative means the deeper residual became decodable at an earlier **training checkpoint**; positive means the shallower residual did. This compares training-time emergence, not the within-forward-pass computation order.

In [ ]:
deltas = adjacent_layer_deltas(events).set_index("target").reindex(target_order)
delta_cols = [column for column in deltas.columns if isinstance(column, str) and column.startswith("L")]
if delta_cols:
    delta_matrix = deltas[delta_cols].astype(float)
    finite = np.abs(delta_matrix.to_numpy()[np.isfinite(delta_matrix.to_numpy())])
    limit = max(float(finite.max()) if len(finite) else 1.0, 1.0)
    fig, ax = plt.subplots(figsize=(1.4 * len(delta_cols) + 3, 0.48 * len(delta_matrix) + 2))
    sns.heatmap(delta_matrix, annot=True, fmt=".0f", cmap="RdBu_r", center=0,
                vmin=-limit, vmax=limit, linewidths=0.5,
                cbar_kws={"label": "deeper − shallower acquisition step"}, ax=ax)
    ax.set_title("Adjacent-residual learning-time differences")
    ax.set_xlabel("adjacent residual pair")
    ax.set_ylabel("teacher latent target")
    fig.tight_layout()
    plt.show()

## 8. Inspect raw accuracy and excess NLL for one target

The ordering uses normalized accuracy, but it should agree qualitatively with raw accuracy approaching the Bayes ceiling and excess NLL approaching zero. A disagreement is a reason to withhold the phase claim and inspect probe sample counts and optimizer behavior.

In [ ]:
detail_level, detail_slot, detail_offset = DETAIL_TARGET
detail = probe_long[(probe_long.level == detail_level) &
                    (probe_long.slot == detail_slot) &
                    (probe_long.offset == detail_offset)]
if detail.empty:
    print(f"DETAIL_TARGET={DETAIL_TARGET} is not present in this run.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
    palette = sns.color_palette("colorblind", spec.num_layers)
    for layer, trajectory in detail.groupby("layer"):
        axes[0].plot(trajectory["_step"], trajectory["acc"],
                     color=palette[layer], linewidth=2, label=f"L{layer}")
        axes[1].plot(trajectory["_step"], trajectory["excess_nll"],
                     color=palette[layer], linewidth=2, label=f"L{layer}")
    bayes = detail.groupby("_step")["bayes_acc"].median().dropna()
    if len(bayes):
        axes[0].plot(bayes.index, bayes.values, color="black", linestyle="--",
                     linewidth=1.5, label="teacher Bayes")
    axes[0].set_ylabel("raw probe accuracy")
    axes[1].set_ylabel("excess NLL (lower is better)")
    for ax in axes:
        ax.set_xlabel("training step")
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].legend(frameon=False, ncol=2)
    fig.suptitle(f"Raw diagnostics: level{detail_level}/slot{detail_slot}/k={detail_offset:+d}")
    fig.tight_layout()
    plt.show()

    final_detail = (detail.sort_values("_step").groupby("layer").tail(1)
                    [["layer", "_step", "acc", "bayes_acc", "acc_progress",
                      "nll", "bayes_nll", "excess_nll", "n"]])
    display(final_detail)

## 9. Optional: aggregate replicate runs

Learning-order claims should be repeated across seeds. Add run IDs below; the cell checks that every run has the same probe layout, then reports median acquisition steps and how often each residual was first.

In [ ]:
REPLICATE_RUN_IDS = []

replicate_events = []
for replicate_id in REPLICATE_RUN_IDS:
    replicate_config = fetch_run_config(replicate_id, entity=ENTITY, project=PROJECT)
    replicate_spec = probe_spec_from_config(replicate_config)
    if replicate_spec != spec:
        raise ValueError(f"{replicate_id} has layout {replicate_spec}, expected {spec}")
    replicate_payload = fetch_run_data(
        replicate_id, probe_metric_keys(spec, PROBE_METRICS, SPLIT),
        entity=ENTITY, project=PROJECT,
    )
    replicate_long = probe_history_long(replicate_payload["df"], spec, SPLIT)
    replicate = acquisition_table(
        replicate_long, threshold=THRESHOLD, min_dwell=MIN_DWELL,
        require_final=REQUIRE_FINAL, offsets=headline_offsets,
    )
    replicate["run_id"] = replicate_id
    replicate_events.append(replicate)

if replicate_events:
    all_events = pd.concat(replicate_events, ignore_index=True)
    median_steps = (all_events.groupby(["target", "layer"])["acquisition_step"]
                    .agg(["median", "count"]).reset_index())
    first = all_events.dropna(subset=["acquisition_step"]).copy()
    first["first_step"] = first.groupby(["run_id", "target"])["acquisition_step"].transform("min")
    first_counts = (first[first.acquisition_step == first.first_step]
                    .groupby(["target", "layer"]).size().rename("times_first").reset_index())
    display(median_steps.merge(first_counts, how="left", on=["target", "layer"])
            .fillna({"times_first": 0}))
else:
    print("Add replicate run IDs to summarize robustness across seeds.")

## Interpretation rules

- Treat `L0` as the learned input/position-encoding baseline, not a transformer block.
- Compare residuals only for the same `(level, slot, offset)` target. Compare teacher levels using uniform-to-Bayes NLL progress, not raw accuracy.
- Keep slots separate: they represent different amounts of within-unit evidence, especially for current-unit refinement.
- Warm-start probes are preferred for emergence timing. SGD probes introduce optimizer lag, so their acquisition steps mix student learning with probe learning.
- Linear decodability is observational, not causal. Nested latent labels are correlated, and a probe does not show where a representation was computed or whether the readout uses it.
- Report the probe cadence, threshold, dwell, and ties. Verify conclusions across seeds and with excess NLL before discussing a stable order of learning.